In [ ]:
from function_lib import *

In [ ]:

class MLEECPParam:
    """
    Exponential-cone ListMLE-style ERM over a finite set Z.

    New problem:

        For each sample n, we have a ranking φ_n over the alternatives z ∈ Z
        based on the true cost vector c_n.

        The optimization variables are:
            - B ∈ R^{d_x × d_c}
            - t_{n,i} for n ∈ [N], i ∈ [K]
            - v_{n,i,k} for n ∈ [N], i ∈ [K], k ∈ {i,...,K-1}

        Objective:
            (1/N) * sum_n [ -sum_i (B^T x_n)^T z_{φ_n(i)} + sum_i t_{n,i} ]

        Constraints:
            ( (B^T x_n)^T z_{φ_n(k)} - t_{n,i},  η,  v_{n,i,k} ) ∈ K_exp
                ∀ n, i, k ∈ {i,...,K-1}

            sum_{k=i}^{K-1} v_{n,i,k} ≤ η,   ∀ n, i

        where K = |Z| and η > 0.
    """

    def __init__(self, N, d_x, d_c, Z, eta: float):
        """
        Parameters
        ----------
        N : int
            Number of samples.
        d_x : int
            Dimension of x_n.
        d_c : int
            Dimension of c_n (and z).
        Z : array, shape (K, d_c)
            All feasible z ∈ Z as rows.
        eta : float
            Temperature / scaling parameter.
        """
        self.N = int(N)
        self.d_x = int(d_x)
        self.d_c = int(d_c)

        self.Z = np.asarray(Z, dtype=float)  # (K, d_c)
        self.eta = float(eta)

        self.K, d_c2 = self.Z.shape
        if d_c2 != self.d_c:
            raise ValueError(
                f"Z must have shape (K, d_c) with d_c={self.d_c}, got {d_c2}"
            )

        # Dimension of flattened B
        self.D = self.d_x * self.d_c

        # Placeholders for last solution
        self.M = None
        self.B_var = None
        self.t_var = None
        self.v_var = None

    # ------------------------------------------------------------------
    # INTERNAL: compute ranking φ_n from C (lexicographic tie-breaking)
    # ------------------------------------------------------------------
    def _compute_phi_from_C(self, C):
        """
        Given cost matrix C ∈ R^{N × d_c} (rows: c_n^T),
        compute ranking φ_n over Z for each n, where

            cost_{n,k} = f(c_n, z_k) = c_n^T z_k

        and φ_n is the permutation that sorts these costs in nondecreasing
        order, with ties broken lexicographically by the original index.

        Returns
        -------
        phi : ndarray, shape (N, K)
            phi[n, i] = index in {0,...,K-1} of the alternative at rank i
            for sample n.
        """
        C = np.asarray(C, dtype=float)
        if C.shape != (self.N, self.d_c):
            raise ValueError(f"C must have shape ({self.N}, {self.d_c})")

        # cost_{n,k} = c_n^T z_k
        costs = C @ self.Z.T  # (N, K)

        # argsort with stable sort => ties broken by original index (0..K-1)
        phi = np.argsort(costs, axis=1, kind="mergesort")  # (N, K)
        return phi

    # ------------------------------------------------------------------
    # PUBLIC: solve ListMLE-ECP for a given dataset (X, C)
    # ------------------------------------------------------------------
    def solve(self, X, C, verbose: bool = True):
        """
        Solve the ListMLE-style ECP problem for dataset (X, C).

        Parameters
        ----------
        X : array, shape (N, d_x)
            Feature matrix (rows = x_n^T).
        C : array, shape (N, d_c)
            Cost matrix (rows = c_n^T); used only to construct rankings φ_n.
        verbose : bool
            If True, show MOSEK log.

        Returns
        -------
        result : dict with keys:
            "B"   : optimal B (d_x, d_c)
            "t"   : optimal t_{n,i} (N, K)
            "v"   : optimal v_{n,i,k} flattened as (N, K, K) with zeros where k < i
            "obj" : optimal objective value
        """
        X = np.asarray(X, dtype=float)
        C = np.asarray(C, dtype=float)

        if X.shape != (self.N, self.d_x):
            raise ValueError(f"X must have shape ({self.N}, {self.d_x})")
        if C.shape != (self.N, self.d_c):
            raise ValueError(f"C must have shape ({self.N}, {self.d_c})")

        # 1. Compute ranking φ_n based on true costs c_n^T z_k
        phi = self._compute_phi_from_C(C)  # (N, K)

        # 2. Build coefficient matrix A for scores:
        #    score_{n,k} = (B^T x_n)^T z_{φ_n(k)}
        #               = <B, x_n z_{φ_n(k)}^T>
        #    where B is flattened row-major.
        N, K = self.N, self.K
        D = self.D

        A = np.zeros((N * K, D), dtype=float)
        for n in range(N):
            x_n = X[n, :]  # (d_x,)
            for k in range(K):
                alt_idx = phi[n, k]  # index of z at rank k
                z_vec = self.Z[alt_idx, :]  # (d_c,)

                # vec(x_n z^T) w.r.t row-major flatten of B: kron(x, z)
                A[n * K + k, :] = np.kron(x_n, z_vec)

        A_mat = Matrix.dense(A)  # (N*K, D)

        # 3. Build MOSEK model
        M = Model("listmle_ecp")
        self.M = M

        # Variables:
        # B as flattened vector (D,)
        Bv = M.variable("B", D, Domain.unbounded())
        self.B_var = Bv

        # t_{n,i} as vector length N*K
        t = M.variable("t", N * K, Domain.unbounded())
        self.t_var = t

        # v_{n,i,k} only for k >= i: we'll store them in a 1D variable of size N * L,
        # where L = number of (i,k) pairs with 0 <= i <= k < K.
        pairs = []
        for i in range(K):
            for k in range(i, K):
                pairs.append((i, k))
        L = len(pairs)

        v = M.variable("v", N * L, Domain.greaterThan(0.0))
        self.v_var = v

        # 4. Scores S = A * Bv, shape (N*K,)
        S = Expr.mul(A_mat, Bv)  # Expression of length N*K

        # 5. Exponential cone constraints:
        #
        # For each (n,i,k) with k >= i:
        #   ( (B^T x_n)^T z_{φ_n(k)} - t_{n,i},  η,  v_{n,i,k} ) ∈ K_exp
        #
        # We'll index:
        #   score-index: idx_s = n*K + k
        #   t-index    : idx_t = n*K + i
        #   v-index    : idx_v = n*L + j, where j indexes (i,k) in 'pairs'.

        for n in range(N):
            for j, (i, k) in enumerate(pairs):
                idx_s = n * K + k
                idx_t = n * K + i
                idx_v = n * L + j

                score_expr = S.index(idx_s)      # scalar expression
                t_expr = t.index(idx_t)          # scalar variable
                v_expr = v.index(idx_v)          # scalar variable

                first_coord = v_expr
                second_coord = self.eta
                third_coord = Expr.sub(score_expr, t_expr)  # S_{n,k} - t_{n,i}

                triplet = Expr.hstack(first_coord, second_coord, third_coord)
                M.constraint(triplet, Domain.inPExpCone())

        # 6. Sum constraints:
        #
        #   sum_{k=i}^{K-1} v_{n,i,k} ≤ η, ∀ n, i.
        #
        # Using the same mapping via pairs and indices.

        # Precompute, for each i, which j in 'pairs' correspond to this i
        j_by_i = [[] for _ in range(K)]
        for j, (i, k) in enumerate(pairs):
            j_by_i[i].append(j)

        for n in range(N):
            for i in range(K):
                v_indices = [n * L + j for j in j_by_i[i]]
                v_terms = [v.index(idx_v) for idx_v in v_indices]
                if v_terms:
                    sum_vi = Expr.add(v_terms[0], v_terms[1:]) if len(v_terms) > 1 else v_terms[0]
                else:
                    # This should not happen since every i appears in at least one pair (i,i)
                    continue

                M.constraint(sum_vi, Domain.lessThan(self.eta))

        # 7. Objective:
        #
        #   (1/N) * sum_n [ -sum_i score_{n,i} + sum_i t_{n,i} ]
        #
        # score_{n,i} = S_{n,i}, with score-index n*K + i
        #
        # We'll build:
        #   obj = (1/N) * [ -sum_{n,i} S_{n,i} + sum_{n,i} t_{n,i} ]

        all_score_indices = [n * K + i for n in range(N) for i in range(K)]
        score_terms = [S.index(idx) for idx in all_score_indices]
        if len(score_terms) > 1:
            sum_scores = Expr.add(score_terms[0], score_terms[1:])
        else:
            sum_scores = score_terms[0]

        all_t_indices = [n * K + i for n in range(N) for i in range(K)]
        t_terms = [t.index(idx) for idx in all_t_indices]
        if len(t_terms) > 1:
            sum_t = Expr.add(t_terms[0], t_terms[1:])
        else:
            sum_t = t_terms[0]

        obj_expr = Expr.sub(sum_t, sum_scores)  # sum t_{n,i} - sum score_{n,i}
        obj_expr = Expr.mul(1.0 / N, obj_expr)

        M.objective("mle_ecp_obj", ObjectiveSense.Minimize, obj_expr)

        # 8. Solve
        if not verbose:
            M.setLogHandler(None)

        M.solve()

        # 9. Extract solution
        B_flat = np.array(Bv.level())  # (D,)
        B_val = B_flat.reshape(self.d_x, self.d_c)

        t_val_flat = np.array(t.level())  # (N*K,)
        t_val = t_val_flat.reshape(N, K)

        v_val_flat = np.array(v.level())  # (N*L,)

        # For convenience, put v into a (N, K, K) array with zeros where k < i
        v_val = np.zeros((N, K, K))
        for n in range(N):
            for j, (i, k) in enumerate(pairs):
                idx_v = n * L + j
                v_val[n, i, k] = v_val_flat[idx_v]

        obj_val = M.primalObjValue()

        return {
            "B": B_val,
            "t": t_val,
            "v": v_val,
            "obj": obj_val,
        }

    # ------------------------------------------------------------------
    # PUBLIC: regret evaluation (unchanged logic)
    # ------------------------------------------------------------------
    def evaluate_regret(self, X_test, C_test, B=None, return_per_sample=False):
        """
        Evaluate the prediction regret on a test dataset (X_test, C_test).

        This part is unchanged: it only uses (X_test, B, Z) and true costs C_test.

        Parameters
        ----------
        X_test : array, shape (N_test, d_x)
        C_test : array, shape (N_test, d_c)
        B : array or None, optional
            If None, uses the current solution stored in the model.
        return_per_sample : bool, optional

        Returns
        -------
        result : dict
        """
        X_test = np.asarray(X_test, dtype=float)
        C_test = np.asarray(C_test, dtype=float)

        N_test, d_x_test = X_test.shape
        N_ctest, d_c_test = C_test.shape

        if d_x_test != self.d_x:
            raise ValueError(f"X_test shape mismatch: expected dim {self.d_x}, got {d_x_test}")
        if d_c_test != self.d_c:
            raise ValueError(f"C_test shape mismatch: expected dim {self.d_c}, got {d_c_test}")
        if N_test != N_ctest:
            raise ValueError("X_test and C_test must have the same number of rows.")

        if B is None:
            if self.B_var is None:
                raise ValueError("No trained B available. Call solve() first or pass B explicitly.")
            B_flat = np.array(self.B_var.level())
            B = B_flat.reshape(self.d_x, self.d_c)
        else:
            B = np.asarray(B, dtype=float).reshape(self.d_x, self.d_c)

        # Predicted costs for all samples
        C_hat = X_test @ B  # (N_test, d_c)

        # Objective values for all feasible z on predicted costs
        obj_pred = C_hat @ self.Z.T  # (N_test, K)

        # Predicted optimal decision index per sample
        idx_hat = np.argmin(obj_pred, axis=1)  # (N_test,)

        # True objective values for all feasible z
        obj_true = C_test @ self.Z.T  # (N_test, K)

        # True cost of the predicted decision
        chosen_costs = obj_true[np.arange(N_test), idx_hat]

        # True cost of the optimal decision (oracle)
        optimal_costs = obj_true.min(axis=1)

        regrets = chosen_costs - optimal_costs
        avg_regret = float(regrets.mean())

        result = {"avg_regret": avg_regret}
        if return_per_sample:
            result["regret_per_sample"] = regrets

        return result


In [ ]:
# ----------------- config -----------------
n_instance   = 1000
input_dim    = 5
noise        = 0.5
deg          = 2
grid_width   = 5
epoch        = 100
learning_rate = 0.001
n_simulation = 50

# ----------------- paths ------------------
base_path = change_to_py_file_dir()
base      = os.path.join(base_path, "Data", "Shortest_path")


In [ ]:

# ----------------- results ----------------
total_result  = {}
# =========================================================
# main loop over seeds
# =========================================================

# ----- build Z (all feasible z) only once -----
sols, edges, G = generate_all_feasible_solutions(grid_width=5)
model_list = ["MLE"]

for model_type in model_list:
    regret_result = []
    parame_result = []
    solver = None  # will build once on the first iteration

    for seed in tqdm(range(n_simulation),
                     desc="Running simulations",
                     colour="green"):

        # ---- reproducibility ----
        random.seed(seed)
        np.random.seed(seed)

        # ---- data for this seed ----
        x_train, y_train, x_test, y_test = load_shortest_path_setting_with_splits(
            base, n_instance, input_dim, deg, noise, grid_width, seed
        )
        
        # ---- build solver only once (reuse the same model) ----
        if solver is None:
            solver = MLEECPParam(
                N=x_train.shape[0],       # must be constant across seeds
                d_x=x_train.shape[1],
                d_c=y_train.shape[1],
                Z=sols,
                eta=0.5
            )

        # ---- solve training problem for this seed ----
        res = solver.solve(x_train, y_train, verbose=False)

        # ---- evaluate regret on test set ----
        # B is already stored in solver.B after solve(), so B argument is optional
        eval_res = solver.evaluate_regret(
            x_test, y_test, return_per_sample=False
        )

        regret_result.append(eval_res["avg_regret"])
        parame_result.append(res["B"])

    # save results for this model type
    total_result[model_type] = {
        "regret": regret_result,
        "params": parame_result
    }
    # ==================================================
    # CRITICAL FIX: Clean up memory before next model!
    # ==================================================
    if solver is not None:
        solver.M.dispose()  # Release Mosek C-memory immediately
        del solver          # Delete Python object
    
    gc.collect()            # Force Python Garbage Collector

    process = psutil.Process(os.getpid())
    print(f"Python process RAM: {process.memory_info().rss/1024/1024/1024:.3f} GB")

# Save the result to pickle file
save_path = os.path.join(base_path, "Results", "Score")
os.makedirs(save_path, exist_ok=True)   # create directory if not exist
file_path = os.path.join(save_path, f"Shortest_path_MLE_result_{n_instance}_{input_dim}_{deg}_{noise}_{grid_width}.pkl")
with open(file_path, "wb") as f:
    pickle.dump(total_result, f)

def print_memory():
    mem = psutil.virtual_memory()
    print(f"Available: {mem.available/1024/1024/1024:.2f} GB")
    print(f"Used:      {(mem.total - mem.available)/1024/1024/1024:.2f} GB")
    print(f"Percent:   {mem.percent}%")

print_memory()

In [ ]:
# Compute stats for the LST model
# regrets = total_result["LST-0.1"]["regret"]

# mean, std, var90, cvar90 = compute_stats(regrets)

# print(f"Mean    : {mean:.6f}")
# print(f"Std     : {std:.6f}")
# print(f"VaR 90  : {var90:.6f}")
# print(f"CVaR 90 : {cvar90:.6f}")
rows = []
for model_type in model_list:
    regrets = total_result[model_type]["regret"]
    mean, std, var90, cvar90 = compute_stats(regrets)

    rows.append({
        "Model": model_type,
        "Mean": mean,
        "Std": std,
        "VaR90": var90,
        "CVaR90": cvar90
    })

# convert to DataFrame
df = pd.DataFrame(rows)

# nicely formatted print
print(df.to_string(index=False, float_format="%.6f"))


# Collect regret distributions across all model types
df_list = []
for model_type, data in total_result.items():
    df_list.append(pd.DataFrame({
        "method": [model_type] * len(data["regret"]),
        "regret": data["regret"]
    }))

df = pd.concat(df_list, ignore_index=True)

# Academic color palette
palette = sns.color_palette("deep")
unique_methods = sorted(df["method"].unique())
method_colors = {m: palette[i % len(palette)] for i, m in enumerate(unique_methods)}

plt.figure(figsize=(6, 4))

sns.boxplot(
    data=df,
    x="method",
    y="regret",
    palette=method_colors,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 5
    }
)

sns.stripplot(
    data=df,
    x="method",
    y="regret",
    color="black",
    alpha=0.25,
    jitter=0.2,
    size=3
)

plt.title("Regret Distribution Across Methods", fontsize=12)
plt.ylabel("Regret")
plt.xlabel("Method")
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()

# Save figure
save_dir = os.path.join(base_path, "Results", "Figures")
os.makedirs(save_dir, exist_ok=True)

plt.savefig(
    os.path.join(save_dir, "regret_comparison.png"),
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# seed = 1
# x_train, y_train, x_test, y_test = load_shortest_path_setting_with_splits(base, n_instance, input_dim, deg, noise, grid_width, seed)

# # Example usage:

# sols, edges, G = generate_all_feasible_solutions(grid_width=5)
# # print("Number of edges:", len(edges))
# # print("Number of feasible s->t paths:", len(sols))
# # print("First solution vector:", sols[0])

# # pick any solution
# # vec = sols[10]
# # draw
# # draw_solution(G, edges, vec, grid_width=5, cost=y_train[0])
# solver = MLEECPParam(
#     N=x_train.shape[0],
#     d_x=x_train.shape[1],
#     d_c=y_train.shape[1],
#     Z=sols,
#     eta=0.5,
#     alpha=1e-4, 
# )

# X_run = x_train   # shape (N, d_x)
# C_run = y_train   # shape (N, d_c)

# res = solver.solve(X_run, C_run, verbose=False)
# eval_res = solver.evaluate_regret(x_test, y_test, return_per_sample=True)
# print("Avg regret:", eval_res["avg_regret"])
# # print("Per-sample regret:", eval_res["regret_per_sample"])
